# Starting Preprocessing

In [101]:
# 0.0 Imports

import pandas as pd
import numpy as np

## Phase 1: Basic Cleanup

**GOAL:**
- to reduce the shape from (1783, 39) -> (1470, 28)

In [102]:
# 1.1 Import the raw dataset

df = pd.read_csv(r'D:\Hustle\Chennai-PG\Data\raw\chennai_pg_dataset.csv')
df.shape

(1783, 39)

In [103]:
# 1.2 Deduplication

"""
Drop duplicate rows corresponding to ID & OCCUPANCY
"""

df = df.drop_duplicates(subset=['id', 'occupancy'])
print(df.shape)
print(df.duplicated().sum())

(1621, 39)
0


In [104]:
# 1.3 Drop columns

"""
Drop columns that are not useful for modeling.

If columns is not provided, the default set of columns
identified during EDA will be removed.

col = ['id', 'title', 'address', 'total_bathrooms', 'warden', 'cooking_allowed', 'gate_closing_time', 'guardian_required', 'nonveg_allowed', 'smoking_allowed'] by phase 1 observation
col = ['lunch', 'breakfast', 'dinner'] by phase 3 observation
"""

df = df.drop(columns=['id', 'title', 'address', 'total_bathrooms', 'warden', 'cooking_allowed', 'gate_closing_time', 'guardian_required', 'nonveg_allowed', 'smoking_allowed', 'lunch', 'breakfast', 'dinner'])
df.shape

(1621, 26)

In [106]:
# 1.4 Drop rows

"""
Drop rows that are not useful for modelling.

- ~1% rows with missing values in key columns
- rent with 0 or NaNs
- occupancy is NaN
- Known confirmed correction for THIS dataset
"""

df = df.dropna(subset=['rent', 'deposit', 'occupancy', 'attached_bathroom'])

# Remove invalid/placeholder rents.
# The minimum realistic PG rent in Chennai is well above 1000.
df = df[df['rent'] >= 1000]

# Known confirmed correction for THIS dataset

# df = df[~((df['deposit'] == 200000) & (df['rent'] == 25000) & (df['locality'] == 'Vadapalani'))]
# df.shape

# handle this in a better way, above logic only for that specific row, this can handle future similar edge cases
DEPOSIT_RENT_RATIO_CAP = 5

df = df[df['deposit'] / df['rent'] <= DEPOSIT_RENT_RATIO_CAP]


In [107]:
# 1.5 fix datatypes & renaming

# Boolean cleanup
"""
FIll missing boolean amenites with False
then convert the columns to bool type
"""
bool_cols = ['attached_bathroom', 'mess', 'wifi', 'laundry', 'power_backup',
        'refrigerator', 'common_tv', 'room_cleaning','room_ac', 
        'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding',
        'room_attached_bath',
]
df[bool_cols] = df[bool_cols].fillna(False).astype(bool)
df['parking'] = df['parking'].fillna('No Parking') # in EDA i actually replaced NaN as none, but it make sense to keep No Parking

# Naming consistency
df['available_for'] = df['available_for'].replace('Both', 'Anyone')


In [108]:
# 1.6 fixes for before doing imputations

# this is for fix the left influend skew-ness (should be done before imputation)
# Convert the invalid sentinel value (-10) to NaN
# before creating missing-value indicators and imputing.
df['transit_score'] = df['transit_score'].replace(-10, np.nan)

# creating tag for msiing values rows
df['transit_score_missing'] = df['transit_score'].isna().astype(int)
df['lifestyle_score_missing'] = df['lifestyle_score'].isna().astype(int)

In [109]:
df.shape

(1468, 28)

## Phase 1 Observation

- our goal (**to reduce the shape from (1783, 39) -> (1470, 28)**) was satisfied

----

# Phase 2: Train/Validation/Test split

**SPLIT PLAN**:

```md
100%
│
├── 70% TRAIN
│
├── 15% VALIDATION
│
└── 15% TEST
```

| Dataset        | Purpose                                                          |
| -------------- | ---------------------------------------------------------------- |
| **Train**      | Learn model parameters + fit preprocessing                       |
| **Validation** | Make decisions: model, hyperparameters, features, encoding, etc. |
| **Test**       | Final unbiased evaluation                                        |


In [110]:
# 2.1 Target and feature

X = df.drop(columns=['rent'])
y = df['rent']

print(X.shape)
print(y.shape)

(1468, 27)
(1468,)


In [111]:
# 2.2 Train/Validation/Test

from sklearn.model_selection import train_test_split

# First spilt (Train = 70%, temp = 30%) here i use validation so, just used temp and then splt the temp -> val/test = 15% each
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=X['occupancy'] # self note: Bug Fix (refer commit description)
)

# Second Split (temp = 30%, split it inro Validation/Train -> 15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=X_temp['occupancy'] # self note: Bug Fix (refer commit description)
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (1027, 27) (1027,)
Validation: (220, 27) (220,)
Test: (221, 27) (221,)


# Phase 2: Results

- Created feature and target variable
- splitted Train/Validation/test as of the plan

---

# Phase 3: Imputations

**GOAL**:

- apply `log1p` for target (rent)
- similarly apply `log1p` transfromation for deposit
- cast bool_cols to int8 (forgoted and added later) skill issue :(
- Impute transit_score and lifestyle_score as per the Phase 2 (EDA) strategy
- `locality`: Smoothed target encoding // **The rule**: always fit target encoding on train only, then apply to val and test.
- `occupancy`: Ordinal encoding
- `gender`, `parking` & `available_for`: One-Hot encoding

In [112]:
# 3.1 Transform rent and deposit
import numpy as np

# For train 
y_train = np.log1p(y_train)
X_train['deposit'] = np.log1p(X_train['deposit'])

# for validation 
y_val = np.log1p(y_val)
X_val['deposit'] = np.log1p(X_val['deposit'])

# for test 
y_test = np.log1p(y_test)
X_test['deposit'] = np.log1p(X_test['deposit'])

In [113]:
# 3.2 Cast bool -> int8

bool_cols = ['attached_bathroom', 'food_included', 'mess', 'wifi', 'laundry', 'power_backup', 'refrigerator', 'common_tv', 'room_cleaning', 'room_ac', 'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding', 'room_attached_bath']

X_train[bool_cols] = X_train[bool_cols].astype('int8')
X_val[bool_cols] = X_val[bool_cols].astype('int8')
X_test[bool_cols] = X_test[bool_cols].astype('int8')

In [ ]:
# 3.3 Transit & lifestle score imptation

# Impute with locality meadian for train/val/test\

# transit_score

# MISTAKE: again i made mistake like what i did in global median , i imputed the val/test median to their set, it should be train's median only
# and i just fixed the global, and never thought of this to check, how careless iam :(
# so replaced the transform(lambda x: x.fillna(x.median)) -> X_val['transit_score'].fillna(X_val['locality'].map(local_transit_median)) for val/test

local_transit_median = X_train.groupby('locality')['transit_score'].median()

X_train['transit_score'] = (
    X_train.groupby('locality')['transit_score']
    .transform(lambda x: x.fillna(x.median())) # can also use the val/test method below using local_transit_median, i just keep the old method for future reference
)
X_val['transit_score'] = (
    X_val['transit_score'].fillna(
        X_val['locality'].map(local_transit_median)
    )
)
X_test['transit_score'] = (
    X_test['transit_score'].fillna(
        X_test['locality'].map(local_transit_median)
    )
)

# lifestyle_score

# mistake, refer the above comments rant, this is only for my future self

local_lifestyle_median = X_train.groupby('locality')['lifestyle_score'].median()

X_train['lifestyle_score'] = (
    X_train.groupby('locality')['lifestyle_score']
    .transform(lambda x: x.fillna(x.median())) # can also use the val/test method below using local_transit_median, i just keep the old method for future reference
)
X_val['lifestyle_score'] = (
    X_val['lifestyle_score'].fillna(
        X_val['locality'].map(local_lifestyle_median)
    )
)
X_test['lifestyle_score'] = (
    X_test['lifestyle_score'].fillna(
        X_test['locality'].map(local_lifestyle_median)
    )
)


# impute with global median (if local median is Nan)
X_train['transit_score'] = X_train['transit_score'].fillna(X_train['transit_score'].median())
X_train['lifestyle_score'] = X_train['lifestyle_score'].fillna(X_train['lifestyle_score'].median())

# MISTAKE: used their own set's median for val/test
# IMPORTANT use X_TRAIN median for val and test, not their own set's median (X_val['transit_score].median()), should be {.fillna(X_train['transit_score'].median())}
X_val['transit_score'] = X_val['transit_score'].fillna(X_train['transit_score'].median())
X_val['lifestyle_score'] = X_val['lifestyle_score'].fillna(X_train['lifestyle_score'].median())

X_test['transit_score'] = X_test['transit_score'].fillna(X_train['transit_score'].median())
X_test['lifestyle_score'] = X_test['lifestyle_score'].fillna(X_train['lifestyle_score'].median())


In [116]:
print(X_train.isna().sum())
print(X_val.isna().sum())
print(X_test.isna().sum())

# fixed my mistake that i only performed imputation only for train datset , so im just did for val/train also

latitude                   0
longitude                  0
locality                   0
gender                     0
available_for              0
transit_score              0
lifestyle_score            0
occupancy                  0
deposit                    0
attached_bathroom          0
food_included              0
mess                       0
wifi                       0
laundry                    0
power_backup               0
refrigerator               0
common_tv                  0
room_cleaning              0
parking                    0
room_ac                    0
room_cupboard              0
room_tv                    0
room_geyser                0
room_bedding               0
room_attached_bath         0
transit_score_missing      0
lifestyle_score_missing    0
dtype: int64
latitude                   0
longitude                  0
locality                   0
gender                     0
available_for              0
transit_score              0
lifestyle_score            0
o

In [118]:
# 3.4 Encoding / column transformation

"""
fit()

    - Look at this data and learn the rules.

transform()

    - Now use those learned rules to convert data.

MENTAL MODEL:
    FIT:                Learn
    TRANSFORM:          Apply
    FIT_TRANSFORM:      learn + apply
    fit only on train
    transform train/val/test
"""

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, TargetEncoder

# 3.4.1 fit and transform on train dataset

# Occupancy -> ordinal encoder
ord_enc = OrdinalEncoder(categories=[['SINGLE','DOUBLE', 'THREE', 'FOUR']]) # self note: plain OrdinalEncoder() will assign categories alphabetically, so i need to define the order explicitly

# fit
ord_enc.fit(X_train[['occupancy']]) # DataFrame should be in → 2D, cuz sklearn expexts 2D not series, df['occupancy'] is a series

# transform
occupancy_train = ord_enc.transform(X_train[['occupancy']]).flatten() # flatten to 1D because occupancy is just one column
# print(f'Ordinal Encoding: \n{ord_enc.categories_}')

# parking, gender & available_for -> onehot encoding
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# fit
ohe.fit(X_train[['gender', 'parking', 'available_for']])

# transform
ohe_train = ohe.transform(X_train[['gender', 'parking', 'available_for']])

ohe.categories_

# locality smoothed target encoding
tar_enc = TargetEncoder(target_type='continuous') # rent is continous so, this rely on target soo

# fit & transform
locality_train = tar_enc.fit_transform(X_train[['locality']], y_train) # x = whcih column to tranform, y = target -> whoch is in y_train soo

In [119]:
# 3.4.2 transform on val/test dataset, apply the learned rules from train to val/test

# Occupancy -> ordinal encoder

# flatten to 1D because occupancy is just one column
occupancy_val = ord_enc.transform(X_val[['occupancy']]).flatten()
occupancy_test = ord_enc.transform(X_test[['occupancy']]).flatten()

# parking, gender, available for -> OHE

ohe_val = ohe.transform(X_val[['gender', 'parking', 'available_for']])

ohe_test = ohe.transform(X_test[['gender', 'parking', 'available_for']])

# locality -> target encoded
"""
fit_transform for train to use cross-fitting
transform only for validation/test
"""

locality_val = tar_enc.transform(X_val[['locality']])
locality_test = tar_enc.transform(X_test[['locality']])

```                   ORIGINAL DATA
                      ORIGINAL DATA
                           │
                           ↓
                  ┌────────────────┐
                  │ Train / Val /  │
                  │      Test      │
                  └────────────────┘
                     │      │     │
                     ↓      ↓     ↓
                   TRAIN    VAL   TEST
                     │
                     │
              LEARN ENCODERS
                     │
        ┌────────────┼─────────────┐
        ↓            ↓             ↓
   OrdinalEncoder   OHE      TargetEncoder
        │            │             │
        │            │             │
      fit()        fit()      fit_transform()
        │            │             │
        ↓            ↓             ↓
     mapping      categories    target stats
        │            │             │
        └────────────┼─────────────┘
                     │
                     ↓
             TRAIN ENCODED
                     │
                     │
          SAME ENCODERS USED
                     │
             ┌───────┴───────┐
             ↓               ↓
           VAL             TEST
        transform()      transform()
             ↓               ↓
       VAL ENCODED      TEST ENCODED

In [39]:
X_train.head()

,latitude,longitude,locality,gender,available_for,transit_score,lifestyle_score,occupancy,deposit,attached_bathroom,...,room_cleaning,parking,room_ac,room_cupboard,room_tv,room_geyser,room_bedding,room_attached_bath,transit_score_missing,lifestyle_score_missing
647,12.913368,80.228812,OMR-Karappakam,FEMALE,Anyone,6.3,6.3,THREE,8.294300,1,...,1,Bike,1,1,1,1,1,1,1,1
853,12.979819,80.242495,Tharamani,MALE,Anyone,8.0,7.9,THREE,8.006701,0,...,0,Bike,1,0,0,0,0,0,0,0
824,12.989589,80.248599,Tharamani,BOTH,Working Professional,7.9,8.3,SINGLE,10.283669,0,...,1,Bike,1,1,0,1,1,0,0,0
212,12.926202,80.111700,Tambaram,MALE,Anyone,8.2,9.6,THREE,7.601402,0,...,0,Bike,0,0,0,0,0,0,0,0
1656,13.053022,80.213763,Vadapalani,BOTH,Anyone,6.7,6.3,THREE,10.308986,0,...,0,Bike,0,0,0,0,0,0,0,0


# Phase 3: Results:

- **all goals defiend in start of phase 3 were satiesfied**

----
# PHASE 4: final feature assembly

GOAL:

```md
Original numerical columns
        +
Occupancy → ordinal
        +
Gender/Parking/Available_for → OHE
        +
Locality → smoothed target encoding
        ↓
FINAL X_train / X_val / X_test
```

In [40]:
X_train.head()

,latitude,longitude,locality,gender,available_for,transit_score,lifestyle_score,occupancy,deposit,attached_bathroom,...,room_cleaning,parking,room_ac,room_cupboard,room_tv,room_geyser,room_bedding,room_attached_bath,transit_score_missing,lifestyle_score_missing
647,12.913368,80.228812,OMR-Karappakam,FEMALE,Anyone,6.3,6.3,THREE,8.294300,1,...,1,Bike,1,1,1,1,1,1,1,1
853,12.979819,80.242495,Tharamani,MALE,Anyone,8.0,7.9,THREE,8.006701,0,...,0,Bike,1,0,0,0,0,0,0,0
824,12.989589,80.248599,Tharamani,BOTH,Working Professional,7.9,8.3,SINGLE,10.283669,0,...,1,Bike,1,1,0,1,1,0,0,0
212,12.926202,80.111700,Tambaram,MALE,Anyone,8.2,9.6,THREE,7.601402,0,...,0,Bike,0,0,0,0,0,0,0,0
1656,13.053022,80.213763,Vadapalani,BOTH,Anyone,6.7,6.3,THREE,10.308986,0,...,0,Bike,0,0,0,0,0,0,0,0


In [120]:
# 4.1 merge transformed (ordinal encoding) occupancy column -> train/test/val splitted datset

X_train['occupancy'] = occupancy_train
X_val['occupancy'] = occupancy_val
X_test['occupancy'] = occupancy_test

# 4.2 merge transformed (OHE) ['gender', 'parking', 'available_for'] column -> train/test/val splitted datset

# get the actual column names OHE created
ohe_cols = ohe.get_feature_names_out(['gender', 'parking', 'available_for'])

# drop the original 3 columns first
X_train = X_train.drop(columns=['gender', 'parking', 'available_for'])
X_val   = X_val.drop(columns=['gender', 'parking', 'available_for'])
X_test  = X_test.drop(columns=['gender', 'parking', 'available_for'])

# add the expanded OHE columns
X_train[ohe_cols] = ohe_train
X_val[ohe_cols]   = ohe_val
X_test[ohe_cols]  = ohe_test

# 4.3 merge transformed (Smoothed target encoding) locality column -> train/test/val splitted datset

X_train['locality'] = locality_train
X_val['locality'] = locality_val
X_test['locality'] = locality_test

# Phase 4 Completed

- and i've finished fit() and transform in train/test/split after tons of mistakes! :(



## SO far all good!!

In [121]:
from pathlib import Path

def save_csv(df: pd.DataFrame | pd.Series, filename):
    output_dir = Path('../Data/processed/modeling')
    output_dir.mkdir(parents=True, exist_ok=True)

    # save preprocessed dataset
    df.to_csv(output_dir / filename, index=False)
    print(f'Preprocessed dataset saved @ {output_dir} as {filename}')

save_csv(X_train, 'X_train.csv')
save_csv(X_val, 'X_val.csv')
save_csv(X_test, 'X_test.csv')

save_csv(y_train, 'y_train.csv')
save_csv(y_val, 'y_val.csv')

save_csv(y_test, 'y_test.csv')

Preprocessed dataset saved @ ..\Data\processed\modeling as X_train.csv
Preprocessed dataset saved @ ..\Data\processed\modeling as X_val.csv
Preprocessed dataset saved @ ..\Data\processed\modeling as X_test.csv
Preprocessed dataset saved @ ..\Data\processed\modeling as y_train.csv
Preprocessed dataset saved @ ..\Data\processed\modeling as y_val.csv
Preprocessed dataset saved @ ..\Data\processed\modeling as y_test.csv


In [ ]:
# helper functions

def missing_info(df):
    missing_info = pd.DataFrame({
        'Missing Count': df.isna().sum(),
        'Percent': (df.isna().mean() * 100).round(2),
        'Data type': df.dtypes,
    })
    
    missing_info = missing_info[missing_info['Missing Count'] > 0]
    print(missing_info.sort_values('Percent',ascending=False))

def is_duplicates(df):
    print(f'Total Duplicates: {df.duplicated().sum()}')

## FINAL CHECK

Before calling preprocessing DONE:

- [ ] No NaN
- [ ] No duplicate rows
- [ ] No categorical strings left
- [ ] Train/val/test have the same feature columns
- [ ] Train/val/test feature order is the same
- [ ] `X` and `y` row counts match
- [ ] No preprocessing value was learned from val/test

In [93]:
missing_info(X_train) # checked for train/test/val
missing_info(X_val)
missing_info(X_test)

,Missing Count,Percent,Data type


In [90]:
is_duplicates(X_train) # checked for train/test/val
is_duplicates(X_val)
is_duplicates(X_test)

Total Duplicates: 0
Total Duplicates: 0
Total Duplicates: 0


In [96]:
X_train.info() # no categorical strs

<class 'pandas.DataFrame'>
Index: 1027 entries, 647 to 1615
Data columns (total 34 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   latitude                            1027 non-null   float64
 1   longitude                           1027 non-null   float64
 2   locality                            1027 non-null   float64
 3   transit_score                       1027 non-null   float64
 4   lifestyle_score                     1027 non-null   float64
 5   occupancy                           1027 non-null   float64
 6   deposit                             1027 non-null   float64
 7   attached_bathroom                   1027 non-null   int8   
 8   food_included                       1027 non-null   int8   
 9   mess                                1027 non-null   int8   
 10  wifi                                1027 non-null   int8   
 11  laundry                             1027 non-null   int8 

In [98]:
print(X_train.columns.nunique()) # train/val/test has same no of feature cols
print(X_test.columns.nunique())
print(X_val.columns.nunique())

34
34
34


In [99]:
print(X_train.columns.tolist()) # train/val/test has same order
print(X_test.columns.tolist())
print(X_val.columns.tolist())

['latitude', 'longitude', 'locality', 'transit_score', 'lifestyle_score', 'occupancy', 'deposit', 'attached_bathroom', 'food_included', 'mess', 'wifi', 'laundry', 'power_backup', 'refrigerator', 'common_tv', 'room_cleaning', 'room_ac', 'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding', 'room_attached_bath', 'transit_score_missing', 'lifestyle_score_missing', 'gender_BOTH', 'gender_FEMALE', 'gender_MALE', 'parking_Bike', 'parking_Bike and Car', 'parking_Car', 'parking_No Parking', 'available_for_Anyone', 'available_for_Student', 'available_for_Working Professional']
['latitude', 'longitude', 'locality', 'transit_score', 'lifestyle_score', 'occupancy', 'deposit', 'attached_bathroom', 'food_included', 'mess', 'wifi', 'laundry', 'power_backup', 'refrigerator', 'common_tv', 'room_cleaning', 'room_ac', 'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding', 'room_attached_bath', 'transit_score_missing', 'lifestyle_score_missing', 'gender_BOTH', 'gender_FEMALE', 'gender_MALE', 'par

In [100]:
print(f'X Rows: {X_train.shape}, y rows: {y_train.shape}') # rows count correct

X Rows: (1027, 34), y rows: (1027,)


# All final checks were performed and verified!!

---
# PREPROCESSING MISTAKE LOG 
---

 **BAsed on the mistakes i've made throught the preprocessing**

 **IMPORTANT SELF NOTE - for my future self**
---
### DO ALL IMPUTATIONS ON TRAIN THEN TRANSFORM THAT TO VAL/TEST

- ❌ First did imputation on the whole `df` before splitting → data leakage.
- ❌ Used `X_val` / `X_test` own medians for global fallback → val/test data was influencing preprocessing.
- ❌ Used `X_val` / `X_test` own locality medians for score imputation → same leakage issue.
- ✅ Fix: calculate all imputation values from `X_train`, then apply those learned values to val/test.

---

## SPLIT

- ❌ Accidentally ended up with `X_train` and `y_train` having different number of rows → `TargetEncoder` threw shape mismatch.
- ✅ Always check `X_train.shape[0] == y_train.shape[0]` after splitting.

---

## ORDINAL ENCODING

- ❌ Initially used plain `OrdinalEncoder()` → categories would be assigned alphabetically.
- ✅ Explicitly defined the real order:
  `SINGLE → DOUBLE → THREE → FOUR`

- ❌ Forgot that `transform()` returns a `(n, 1)` array for one column.
- ✅ Used `.flatten()` before putting it back into the single pandas column.

---

## ONE-HOT ENCODING

- ❌ Called `.fit()` separately for `gender`, `parking`, and `available_for` → each `.fit()` overwrote the previous one.
- ❌ Accidentally swapped `gender` and `parking` during transform 💀
- ✅ Fit once on all 3 columns, then transform all 3 together.

- ❌ Tried assigning the OHE output directly to the original 3 columns.
- ✅ OHE creates NEW columns, so create the encoded columns and merge them, then drop the originals.

---

## TARGET ENCODING

- ❌ Initially confused `target_type` with the type of encoding.
- ✅ `rent` is continuous → `target_type='continuous'`.

- ❌ Target encoding must NOT learn from val/test.
- ✅ Fit/fit_transform using `X_train + y_train`, then only `.transform()` val/test.

---

## OUTLIER / BAD ROW HANDLING

- ❌ Hardcoded one exact bad row:
  `deposit=200000, rent=25000, locality='Vadapalani'`
- ⚠️ This only catches that exact row and won't work properly when new scraped data comes in.
- ✅ Use the actual bad-data pattern/ratio instead of depending on exact values.

---

# **Preprocessing Workflow Reference**

A step by step reference for building a clean, leak free preprocessing pipeline.
Written from real mistakes made during the Chennai PG Intelligence project.

---

## The One Rule That Governs Everything

> **Split first. Transform after. Always fit on train only.**

If you remember nothing else from this document, remember that. Every mistake in preprocessing comes back to violating this rule in some form.

---

## The Full Flow at a Glance

```md
RAW DATA
    │
    ▼
PHASE 1: Basic Cleanup          ← operates on the full df, before splitting
    │   deduplication
    │   drop useless columns
    │   drop invalid rows
    │   fix dtypes
    │   rename for consistency
    │   fix sentinel values (e.g. -10 → NaN)
    │   create indicator flags (e.g. score_missing)
    │
    ▼
PHASE 2: Split                  ← the most important step
    │   X / y separation
    │   Train / Val / Test split (stratified if needed)
    │
    ▼
PHASE 3: Transformations        ← fit on train only, apply to all three
    │   log transforms on target and skewed numerics
    │   cast booleans to int
    │   impute missing values (locality median → global median fallback)
    │   encode categoricals (ordinal, one hot, target)
    │
    ▼
PHASE 4: Final Assembly         ← merge all encoded columns back
    │   assign ordinal encoded column back
    │   drop original OHE columns, add expanded OHE columns
    │   assign target encoded column back
    │
    ▼
PHASE 5: Sanity Checks          ← verify everything before saving
    │   no NaN anywhere
    │   no duplicates
    │   no categorical strings remaining
    │   same columns in train/val/test
    │   same column order in train/val/test
    │   X and y row counts match
    │   nothing was learned from val or test
    │
    ▼
SAVE                            ← only after all checks pass
```

---

## Phase 1: Basic Cleanup

This phase runs on the full raw dataframe before any split. Nothing here learns from the data in a way that leaks — it is purely structural fixes and obviously bad rows.

### 1.1 Deduplication

Always deduplicate on the natural composite key, not on all columns blindly.

```python
df = df.drop_duplicates(subset=['id', 'occupancy'], keep='first')
```

Why composite key: a property with multiple room types (Single, Double, etc.) is not a duplicate — only the same property with the same room type is.

### 1.2 Drop Columns

Drop columns that carry no predictive signal or are structurally redundant. Document why each one was dropped.

```python
df = df.drop(columns=[
    'id',             # identifier, no signal
    'title',          # free text, redundant with locality
    'address',        # redundant with locality
    'gate_closing_time',  # 80%+ missing, low signal
    'total_bathrooms',    # erroneous values, redundant
])
```

### 1.3 Drop Invalid Rows

```python
# drop rows where target or key features are missing
df = df.dropna(subset=['rent', 'deposit', 'occupancy', 'attached_bathroom'])

# drop physically impossible target values
df = df[df['rent'] >= 1000]

# drop rows with implausible feature relationships (ratio based, not hardcoded)
DEPOSIT_RENT_RATIO_CAP = 5
df = df[df['deposit'] / df['rent'] <= DEPOSIT_RENT_RATIO_CAP]
```

**Never hardcode a specific bad row like this:**
```python
# WRONG — only catches one known row, misses future similar errors
df = df[~((df['deposit'] == 200000) & (df['rent'] == 25000))]
```

**Always use a rule that describes the actual problem, not the specific symptom.**

### 1.4 Fix Dtypes and Rename

```python
# fill missing booleans before casting
bool_cols = ['wifi', 'laundry', 'power_backup', ...]
df[bool_cols] = df[bool_cols].fillna(False).astype(bool)

# fill categorical missingness with a real label
df['parking'] = df['parking'].fillna('No Parking')

# rename ambiguous category labels
df['available_for'] = df['available_for'].replace('Both', 'Anyone')
```

### 1.5 Fix Sentinel Values and Create Indicator Flags

Do this before any split. Sentinel values are invalid numbers used to mean "missing" (like -10 for a score that can't be negative).

```python
# replace sentinel with real NaN
df['transit_score'] = df['transit_score'].replace(-10, np.nan)

# create indicator flags BEFORE imputation — captures the original missingness pattern
df['transit_score_missing'] = df['transit_score'].isna().astype(int)
df['lifestyle_score_missing'] = df['lifestyle_score'].isna().astype(int)
```

**Why create the indicator flag:** the fact that a score was missing may itself be predictive. If you impute first and then try to create the flag, the flag will always be zero — you've already filled the NaNs.

---

## Phase 2: Split

This is the most important phase. Everything that "learns" from the data must happen after this point, using only the train set.

### 2.1 Separate X and y

```python
X = df.drop(columns=['rent'])
y = df['rent']
```

### 2.2 Train / Val / Test Split

The standard approach for medium sized datasets:

```md
Train  70%   model learns from this
Val    15%   model selection, hyperparameter tuning, feature decisions
Test   15%   final evaluation only, touched once at the very end
```

```python
from sklearn.model_selection import train_test_split

# first split: train vs temp (val + test combined)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=X['occupancy']   # stratify if a key column is imbalanced
)

# second split: temp into val and test equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=X_temp['occupancy']
)
```

**When to stratify:** when a column that strongly drives the target is significantly imbalanced in the dataset. In this project, SINGLE occupancy is only 13% of rows — without stratification, a random split might give val/test very few SINGLE rooms.

**When not to stratify:** when all categories are naturally balanced. Stratifying on a balanced column adds complexity for no benefit.

---

## Phase 3: Transformations

From this point forward, the rule is:

```md
FIT    →  train only
TRANSFORM  →  train, val, and test
```

Every transformation that "learns" something (a median, a mapping, a category list, a target mean) must learn it from train data only, then apply that learned value to val and test.

### 3.1 Log Transform (Skewed Numerics and Target)

Apply log1p when a column is heavily right skewed. log1p handles zero values safely (unlike log).

```python
# target
y_train = np.log1p(y_train)
y_val   = np.log1p(y_val)
y_test  = np.log1p(y_test)

# skewed feature
X_train['deposit'] = np.log1p(X_train['deposit'])
X_val['deposit']   = np.log1p(X_val['deposit'])
X_test['deposit']  = np.log1p(X_test['deposit'])
```

Note: log1p is applied to all three splits without fitting — it is a fixed mathematical transform, not a learned one. No leakage risk here.

When predicting later, reverse with:

```python
predicted_rent = np.expm1(model.predict(X_test))
```

### 3.2 Cast Booleans to Int

Tree models handle booleans fine, but most sklearn estimators and linear models expect numeric input.

```python
bool_cols = ['wifi', 'laundry', 'food_included', ...]

X_train[bool_cols] = X_train[bool_cols].astype('int8')
X_val[bool_cols]   = X_val[bool_cols].astype('int8')
X_test[bool_cols]  = X_test[bool_cols].astype('int8')
```

### 3.3 Imputing Missing Values

The pattern is always the same: compute the fill value from train, then use that same value to fill val and test.

**Two level imputation (locality median, then global median fallback):**

```python
# step 1: compute from train only
local_transit_median = X_train.groupby('locality')['transit_score'].median()
global_transit_median = X_train['transit_score'].median()

# step 2: apply locality median to all three splits
# IMPORTANT: always use X['locality'].map(...) not X['score_col'].map(...)
# the median dict is keyed by locality name, not by score value

X_train['transit_score'] = X_train['transit_score'].fillna(
    X_train['locality'].map(local_transit_median)
)
X_val['transit_score'] = X_val['transit_score'].fillna(
    X_val['locality'].map(local_transit_median)   # same train medians, val's locality as key
)
X_test['transit_score'] = X_test['transit_score'].fillna(
    X_test['locality'].map(local_transit_median)  # same train medians, test's locality as key
)

# step 3: global fallback for any locality not seen in train
X_train['transit_score'] = X_train['transit_score'].fillna(global_transit_median)
X_val['transit_score']   = X_val['transit_score'].fillna(global_transit_median)
X_test['transit_score']  = X_test['transit_score'].fillna(global_transit_median)
```

**The most common imputation mistake:**

```python
# WRONG — val/test using their own data to fill themselves
X_val['transit_score'] = X_val['transit_score'].fillna(X_val['transit_score'].median())

# WRONG — mapping against the score column instead of locality
X_val['transit_score'] = X_val['transit_score'].fillna(
    X_val['transit_score'].map(local_transit_median)  # float values can't match locality keys
)

# CORRECT
X_val['transit_score'] = X_val['transit_score'].fillna(
    X_val['locality'].map(local_transit_median)
)
```

**Why the wrong version is dangerous:** it gives val/test their own data's median instead of what the model learned from train. This is a subtle form of data leakage. It often doesn't crash and the NaN check still passes (because NaN count is still zero), so you never catch it unless you read the code carefully.

### 3.4 Encoding Categorical Columns

#### Ordinal Encoding

Use when categories have a genuine, meaningful order.

```python
from sklearn.preprocessing import OrdinalEncoder

# always define the order explicitly — do not rely on alphabetical default
ord_enc = OrdinalEncoder(categories=[['SINGLE', 'DOUBLE', 'THREE', 'FOUR']])

# fit on train only
ord_enc.fit(X_train[['occupancy']])

# transform all three — flatten because it returns (n, 1) for a single column
occupancy_train = ord_enc.transform(X_train[['occupancy']]).flatten()
occupancy_val   = ord_enc.transform(X_val[['occupancy']]).flatten()
occupancy_test  = ord_enc.transform(X_test[['occupancy']]).flatten()
```

#### One Hot Encoding

Use when categories have no natural order and cardinality is low (under ~15 categories).

```python
from sklearn.preprocessing import OneHotEncoder

# fit all columns in one call — never fit each column separately
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[['gender', 'parking', 'available_for']])

# transform
ohe_train = ohe.transform(X_train[['gender', 'parking', 'available_for']])
ohe_val   = ohe.transform(X_val[['gender', 'parking', 'available_for']])
ohe_test  = ohe.transform(X_test[['gender', 'parking', 'available_for']])
```

**Never fit each column separately.** Calling `.fit()` a second time overwrites what the first fit learned. You'll end up with the encoder only knowing about the last column you fitted it on.

#### Smoothed Target Encoding

Use when cardinality is high (many unique values) and some categories have very few rows.

```python
from sklearn.preprocessing import TargetEncoder

tar_enc = TargetEncoder(target_type='continuous', smooth=20)

# fit_transform on train — uses cross-fitting internally to avoid within-train leakage
locality_train = tar_enc.fit_transform(X_train[['locality']], y_train)

# transform only for val/test — never fit on val/test
locality_val  = tar_enc.transform(X_val[['locality']])
locality_test = tar_enc.transform(X_test[['locality']])
```

**Why fit_transform on train and transform only on val/test:** target encoding uses the target variable to create the feature. If it saw val/test targets during fitting, it would have future information about the thing it's trying to predict. Always keep it train only.

---

## Phase 4: Final Assembly

Merge all the encoded columns back into the main DataFrames.

```python
# ordinal — single column, assign directly
X_train['occupancy'] = occupancy_train
X_val['occupancy']   = occupancy_val
X_test['occupancy']  = occupancy_test

# OHE — expands original columns into new ones, so drop originals first
ohe_cols = ohe.get_feature_names_out(['gender', 'parking', 'available_for'])

X_train = X_train.drop(columns=['gender', 'parking', 'available_for'])
X_val   = X_val.drop(columns=['gender', 'parking', 'available_for'])
X_test  = X_test.drop(columns=['gender', 'parking', 'available_for'])

X_train[ohe_cols] = ohe_train
X_val[ohe_cols]   = ohe_val
X_test[ohe_cols]  = ohe_test

# target encoding — single column, assign directly
X_train['locality'] = locality_train
X_val['locality']   = locality_val
X_test['locality']  = locality_test
```

**Why drop before adding OHE columns:** OHE takes 3 columns and produces 10 (or however many categories exist across all three). You cannot shove 10 columns back into 3 column slots. Drop the originals, then add the new expanded set.

---

## Phase 5: Sanity Checks

Run all of these before saving. If any check fails, go back and fix it. Do not save a broken dataset.

```python
# 1. no NaN anywhere
assert X_train.isna().sum().sum() == 0, "NaN found in X_train"
assert X_val.isna().sum().sum()   == 0, "NaN found in X_val"
assert X_test.isna().sum().sum()  == 0, "NaN found in X_test"

# 2. no duplicates
assert X_train.duplicated().sum() == 0
assert X_val.duplicated().sum()   == 0
assert X_test.duplicated().sum()  == 0

# 3. no categorical strings remaining — all columns should be numeric
assert all(X_train.dtypes != 'object'), "String columns still present"

# 4 and 5. same columns in same order across all three splits
assert X_train.columns.tolist() == X_val.columns.tolist() == X_test.columns.tolist()

# 6. X and y row counts match
assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0]   == y_val.shape[0]
assert X_test.shape[0]  == y_test.shape[0]

print("All checks passed.")
```

The column order check (check 5) is the one most people skip. If train and test have the same columns but in a different order, a model trained on train will silently predict against the wrong features on test. No error, just wrong predictions.

---

## Save — After Checks Pass

```python
from pathlib import Path

def save_split(df, filename, output_dir='Data/processed/modeling'):
    path = Path(output_dir)
    path.mkdir(parents=True, exist_ok=True)
    df.to_csv(path / filename, index=False)
    print(f"Saved: {filename}  {df.shape}")

save_split(X_train, 'X_train.csv')
save_split(X_val,   'X_val.csv')
save_split(X_test,  'X_test.csv')
save_split(y_train, 'y_train.csv')
save_split(y_val,   'y_val.csv')
save_split(y_test,  'y_test.csv')
```

---

## Common Mistakes Reference

| Mistake | Why it's wrong | Fix |
|---|---|---|
| Impute before splitting | val/test information leaks into the imputed values | split first, impute after |
| Use val/test's own median as fallback | val/test are influencing their own preprocessing | always use X_train's median |
| Map score column instead of locality column during imputation | score values can't match locality name keys, silently returns NaN | use X_split['locality'].map(median_dict) |
| Fit OHE separately per column | each fit() call overwrites the previous | fit once on all columns together |
| Assign OHE output back to original columns | OHE expands columns, you can't fit more into fewer | drop originals, add expanded columns |
| Use plain OrdinalEncoder() without defining order | categories assigned alphabetically by default | always pass categories=[['your', 'order', 'here']] |
| Forget flatten() on ordinal/ordinal encoded output | transform() returns (n, 1) not (n,) | .flatten() before assigning to a single column |
| Fit TargetEncoder on val/test | target variable from val/test leaks into the feature | fit_transform on train, transform only on val/test |
| Hardcode a specific bad row to drop | only catches that one row, misses future similar errors | use a ratio or rule that describes the actual problem |
| Save before sanity checks | might save a corrupted dataset | run all checks, then save |

---

## Encoding Decision Guide

| Situation | Encoding | Example |
|---|---|---|
| Categories have a natural order | Ordinal | occupancy (Single, Double, Three, Four) |
| No order, low cardinality (under 15) | One Hot | gender, parking, available_for |
| No order, high cardinality, some rare categories | Smoothed Target | locality (36 categories, long tail) |
| Already binary (True/False) | Cast to int | all boolean amenity columns |
| 2 categories only | One Hot with drop='first' or binary | yes/no columns |

---

## The Mental Model for Fit vs Transform

```md
TRAIN DATA
    │
    ▼
  fit()          ← encoder looks at train and learns the rules
    │               ordinal: what order?
    │               ohe: what categories exist?
    │               target: what is the mean rent per locality?
    │               imputer: what is the median per locality?
    ▼
transform()      ← encoder applies the learned rules to train
    │
    ├──► transform val    ← same rules, applied to val
    │
    └──► transform test   ← same rules, applied to test
```

Val and test never teach the encoder anything. They only receive the rules that train taught.